In [3]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score

In [4]:
DATA_START_SEASON = 2009
MODEL_START_SEASON = 2010
VAL_SEASON = 2024
TEST_SEASON = 2025
SEASONS = list(range(DATA_START_SEASON, TEST_SEASON + 1))


In [5]:
pbp_columns = [
    'game_id',
    'posteam',
    'defteam',
    'epa',
    'success',
    'pass',
    'rush',
    'qb_kneel',
    'qb_spike',
    'two_point_attempt',
    'interception',
    'fumble_lost',
    'sack',
    'qb_dropback',
    'pass_attempt',
    'qb_hit',
    'passer_player_id',
    'passer_player_name',
]

schedule_columns = [
    'game_id',
    'season',
    'week',
    'gameday',
    'game_type',
    'home_team',
    'away_team',
    'home_score',
    'away_score',
    'home_rest',
    'away_rest',
    'location',
    'home_qb_id',
    'away_qb_id',
    'home_qb_name',
    'away_qb_name',
]

In [6]:
pbp = nfl.load_pbp(SEASONS).select(pbp_columns).to_pandas()
schedule = nfl.load_schedules(SEASONS).select(schedule_columns).to_pandas()

In [7]:
TEAM_ABBR_MAP = {
    "SD": "LAC",
    "OAK": "LV",
    "STL": "LA",
}
pbp[["posteam", "defteam"]] = (
    pbp[["posteam", "defteam"]]
    .replace(TEAM_ABBR_MAP)
)
schedule[["home_team", "away_team"]] = (
    schedule[["home_team", "away_team"]]
    .replace(TEAM_ABBR_MAP)
)

In [8]:
scrimmage_plays = pbp.loc[
    pbp['posteam'].notna()
    & pbp['defteam'].notna()
    & pbp['epa'].notna()
    & pbp['success'].notna()
    & (pbp['pass'].eq(1) | pbp['rush'].eq(1))
    & pbp['qb_kneel'].fillna(0).ne(1)
    & pbp['qb_spike'].fillna(0).ne(1)
    & pbp['two_point_attempt'].fillna(0).ne(1)
].copy()

In [9]:
offense = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        off_epa = ('epa', 'mean'),
        off_success_rate = ('success', 'mean')
    )
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

In [10]:
passing = (
    scrimmage_plays.loc[scrimmage_plays["pass"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(pass_epa = ('epa', 'mean'))
    .reset_index()
    .rename(columns={'posteam': 'team'})
)
rush = (
    scrimmage_plays.loc[scrimmage_plays["rush"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(rush_epa = ('epa', 'mean'))
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

In [11]:
defense = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        def_epa_allowed = ('epa', 'mean'),
        def_success_rate_allowed = ('success', 'mean'),
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)

In [12]:
home_points = (
    schedule[['game_id', 'home_team', 'home_score', 'away_score']]
    .rename(columns={'home_team': 'team', 'home_score': 'points_scored', 'away_score': 'points_allowed'})
)
away_points = (
    schedule[['game_id', 'away_team', 'home_score', 'away_score']]
    .rename(columns={'away_team': 'team', 'home_score': 'points_allowed', 'away_score': 'points_scored'})
)
points = pd.concat([home_points, away_points], ignore_index=True)
points.sort_values(by=['game_id', 'team'], inplace=True)
points.reset_index(drop=True, inplace=True)


In [13]:
points['win'] = np.select(
    [
        points['points_scored'] > points['points_allowed'],
        points['points_scored'] < points['points_allowed'],
    ],
    [1.0, 0.0],
    default=0.5,
)

In [14]:
scrimmage_plays['turnover'] = (
    scrimmage_plays['interception'].fillna(0)
    + scrimmage_plays['fumble_lost'].fillna(0)
).clip(upper=1)

In [15]:
off_turnovers = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(turnovers = ('turnover', 'sum'))
).reset_index().rename(columns={'posteam': 'team'})

In [16]:
def_takeaways = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(takeaways = ('turnover', 'sum')
).reset_index().rename(columns={'defteam': 'team'}))

In [17]:
off_pass_protection = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        sacks_allowed = ('sack', 'sum'),
        dropbacks = ('qb_dropback', 'sum'))
).reset_index().rename(columns={'posteam': 'team'})
off_pass_protection['off_sack_rate'] = (
    off_pass_protection['sacks_allowed'] / off_pass_protection['dropbacks']
)

In [18]:
def_sack_rate = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        sacks = ('sack', 'sum'),
        dropbacks = ('qb_dropback', 'sum'))
).reset_index().rename(columns={'defteam': 'team'})
def_sack_rate['def_sack_rate'] = (
    def_sack_rate['sacks'] / def_sack_rate['dropbacks']
)

In [19]:
completed_schedule = schedule.loc[
    schedule['home_score'].notna()
    & schedule['away_score'].notna()
].copy()
completed_schedule['neutral_site'] = (
    completed_schedule['location']
    .astype(str)
    .str.lower()
    .eq('neutral')
    .astype(int)
)
base_columns = [
    'season',
    'week',
    'gameday',
    'game_id',
    'neutral_site',
]
home_rows = completed_schedule[base_columns].copy()
home_rows['team'] = completed_schedule['home_team']
home_rows['opponent'] = completed_schedule['away_team']
home_rows['rest_days'] = completed_schedule['home_rest']
home_rows['is_home'] = 1
away_rows = completed_schedule[base_columns].copy()
away_rows['team'] = completed_schedule['away_team']
away_rows['opponent'] = completed_schedule['home_team']
away_rows['rest_days'] = completed_schedule['away_rest']
away_rows['is_home'] = 0
schedule_team_games_df = pd.concat([home_rows, away_rows], ignore_index=True)

In [20]:
features_df = [
    offense,
    passing,
    rush,
    defense,
    points,
    off_turnovers,
    def_takeaways,
    off_pass_protection[['game_id', 'team', 'off_sack_rate']],
    def_sack_rate[['game_id', 'team', 'def_sack_rate']],
]
team_games_df = schedule_team_games_df.copy()
for feature_df in features_df:
    assert not feature_df.duplicated(
        ["game_id", "team"]
    ).any()

    team_games_df = team_games_df.merge(
        feature_df,
        on=["game_id", "team"],
        how="left",
        validate="one_to_one",
    )

In [21]:
team_game_columns = [
    'season',
    'week',
    'gameday',
    'game_id',
    'team',
    'opponent',
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
    'points_scored',
    'points_allowed',
    'win',
    'turnovers',
    'takeaways',
    'off_sack_rate',
    'def_sack_rate',
    'rest_days',
    'is_home',
    'neutral_site']
team_games_df = (
    team_games_df[team_game_columns]
    .sort_values(by=['gameday', 'game_id', 'team'])
    .reset_index(drop=True))
team_games_df.head()

,season,week,gameday,game_id,team,opponent,off_epa,off_success_rate,pass_epa,rush_epa,...,points_scored,points_allowed,win,turnovers,takeaways,off_sack_rate,def_sack_rate,rest_days,is_home,neutral_site
0,2009,1,2009-09-10,2009_01_TEN_PIT,PIT,TEN,-0.143463,0.388889,0.034029,-0.546853,...,13,10,1.0,3.0,2.0,0.083333,0.027778,7,1,0
1,2009,1,2009-09-10,2009_01_TEN_PIT,TEN,PIT,-0.034998,0.359375,0.052764,-0.171905,...,10,13,0.0,2.0,3.0,0.027778,0.083333,7,0,0
2,2009,1,2009-09-13,2009_01_CHI_GB,CHI,GB,-0.237903,0.380282,-0.275556,-0.189319,...,15,21,0.0,4.0,0.0,0.054054,0.125000,7,0,0
3,2009,1,2009-09-13,2009_01_CHI_GB,GB,CHI,-0.139636,0.370370,-0.064642,-0.248718,...,21,15,1.0,0.0,4.0,0.125000,0.054054,7,1,0
4,2009,1,2009-09-13,2009_01_DAL_TB,DAL,TB,0.342538,0.423077,0.656938,-0.053878,...,34,21,1.0,0.0,0.0,0.034483,0.000000,7,0,0


In [22]:
recent_metrics = [
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
]

In [23]:
for column in recent_metrics:
    team_games_df[f'{column}_last3'] = (
        team_games_df
        .groupby(['season', 'team'])[column]
        .transform(
            lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
            )

In [24]:
for column in recent_metrics:
    team_games_df[f'{column}_last5'] = (
        team_games_df
        .groupby(['season', 'team'])[column]
        .transform(
            lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
            )

In [25]:
EWM_SPAN = 5
for column in recent_metrics:
    team_games_df[f'{column}_ewm'] = (
        team_games_df
        .groupby(['season', 'team'])[column]
        .transform(
            lambda x: x.shift(1).ewm(
                span=EWM_SPAN, adjust=False, min_periods=1).mean()
            )
        )
team_games_df.head()

,season,week,gameday,game_id,team,opponent,off_epa,off_success_rate,pass_epa,rush_epa,...,pass_epa_last5,rush_epa_last5,def_epa_allowed_last5,def_success_rate_allowed_last5,off_epa_ewm,off_success_rate_ewm,pass_epa_ewm,rush_epa_ewm,def_epa_allowed_ewm,def_success_rate_allowed_ewm
0,2009,1,2009-09-10,2009_01_TEN_PIT,PIT,TEN,-0.143463,0.388889,0.034029,-0.546853,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009,1,2009-09-10,2009_01_TEN_PIT,TEN,PIT,-0.034998,0.359375,0.052764,-0.171905,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,1,2009-09-13,2009_01_CHI_GB,CHI,GB,-0.237903,0.380282,-0.275556,-0.189319,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009,1,2009-09-13,2009_01_CHI_GB,GB,CHI,-0.139636,0.370370,-0.064642,-0.248718,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009,1,2009-09-13,2009_01_DAL_TB,DAL,TB,0.342538,0.423077,0.656938,-0.053878,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
prior_metrics = [
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
]

In [27]:
previous_season = (
    team_games_df
    .groupby(['season', 'team'])[prior_metrics]
    .mean()
    .reset_index()
)
previous_season['season'] += 1

previous_season = previous_season.rename(
    columns={
        column: f"{column}_previous_season" for column in prior_metrics
        }
    )

In [28]:
team_games_df = team_games_df.merge(
    previous_season,
    on=['season', 'team'],
    how='left',
)

In [29]:
league_previous = (
    team_games_df
    .groupby('season', as_index=False)[prior_metrics]
    .mean()
)
league_previous['season'] += 1
league_previous = league_previous.rename(
    columns = {
        metric: f'league_{metric}_previous'
        for metric in prior_metrics
    }
)

In [30]:
team_games_df = team_games_df.merge(
    league_previous,
    on='season',
    how='left',
)

In [31]:
team_games_df.columns

Index(['season', 'week', 'gameday', 'game_id', 'team', 'opponent', 'off_epa',
       'off_success_rate', 'pass_epa', 'rush_epa', 'def_epa_allowed',
       'def_success_rate_allowed', 'points_scored', 'points_allowed', 'win',
       'turnovers', 'takeaways', 'off_sack_rate', 'def_sack_rate', 'rest_days',
       'is_home', 'neutral_site', 'off_epa_last3', 'off_success_rate_last3',
       'pass_epa_last3', 'rush_epa_last3', 'def_epa_allowed_last3',
       'def_success_rate_allowed_last3', 'off_epa_last5',
       'off_success_rate_last5', 'pass_epa_last5', 'rush_epa_last5',
       'def_epa_allowed_last5', 'def_success_rate_allowed_last5',
       'off_epa_ewm', 'off_success_rate_ewm', 'pass_epa_ewm', 'rush_epa_ewm',
       'def_epa_allowed_ewm', 'def_success_rate_allowed_ewm',
       'off_epa_previous_season', 'off_success_rate_previous_season',
       'pass_epa_previous_season', 'rush_epa_previous_season',
       'def_epa_allowed_previous_season',
       'def_success_rate_allowed_previous_

In [32]:
PRIOR_TEAM_WEIGHT = 0.70

In [33]:
print(prior_metrics)

['off_epa', 'off_success_rate', 'pass_epa', 'rush_epa', 'def_epa_allowed', 'def_success_rate_allowed']


In [34]:
for metric in prior_metrics:
    team_previous_column = f'{metric}_previous_season'
    league_previous_column = f'league_{metric}_previous'
    prior_column = f'{metric}_prior'

    team_previous_value = team_games_df[team_previous_column].fillna(
        team_games_df[league_previous_column]
    )

    team_games_df[prior_column] = (
        PRIOR_TEAM_WEIGHT * team_previous_value
        + (1 - PRIOR_TEAM_WEIGHT) * team_games_df[league_previous_column]
    )

In [35]:
team_games_df['pregame_off_epa_current'] = (
    team_games_df
    .groupby(['season', 'team'])['off_epa']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['pregame_off_success_rate_current'] = (
    team_games_df
    .groupby(['season', 'team'])['off_success_rate']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['pregame_pass_epa_current'] = (
    team_games_df
    .groupby(['season', 'team'])['pass_epa']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['pregame_rush_epa_current'] = (
    team_games_df
    .groupby(['season', 'team'])['rush_epa']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['pregame_def_epa_allowed_current'] = (
    team_games_df
    .groupby(['season', 'team'])['def_epa_allowed']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['pregame_def_success_rate_allowed_current'] = (
    team_games_df
    .groupby(['season', 'team'])['def_success_rate_allowed']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['prior_games'] = (
    team_games_df
    .groupby(['season', 'team'])
    .cumcount()
)

In [36]:
PRIOR_GAME_WEIGHT = 4

In [37]:
team_games_df['pregame_off_epa_blended'] = (
    (PRIOR_GAME_WEIGHT * team_games_df['off_epa_prior']
      + team_games_df['prior_games'] * team_games_df['pregame_off_epa_current'].fillna(0)
    )
    / (PRIOR_GAME_WEIGHT + team_games_df['prior_games'])
)
team_games_df['pregame_off_success_rate_blended'] = (
    (PRIOR_GAME_WEIGHT * team_games_df['off_success_rate_prior']
      + team_games_df['prior_games'] * team_games_df['pregame_off_success_rate_current'].fillna(0)
    )
    / (PRIOR_GAME_WEIGHT + team_games_df['prior_games'])
)
team_games_df['pregame_pass_epa_blended'] = (
    (PRIOR_GAME_WEIGHT * team_games_df['pass_epa_prior']
      + team_games_df['prior_games'] * team_games_df['pregame_pass_epa_current'].fillna(0)
    )
    / (PRIOR_GAME_WEIGHT + team_games_df['prior_games'])
)
team_games_df['pregame_rush_epa_blended'] = (
    (PRIOR_GAME_WEIGHT * team_games_df['rush_epa_prior']
      + team_games_df['prior_games'] * team_games_df['pregame_rush_epa_current'].fillna(0)
    )
    / (PRIOR_GAME_WEIGHT + team_games_df['prior_games'])
)
team_games_df['pregame_def_epa_allowed_blended'] = (
    (PRIOR_GAME_WEIGHT * team_games_df['def_epa_allowed_prior']
      + team_games_df['prior_games'] * team_games_df['pregame_def_epa_allowed_current'].fillna(0)
    )
    / (PRIOR_GAME_WEIGHT + team_games_df['prior_games'])
)
team_games_df['pregame_def_success_rate_allowed_blended'] = (
    (PRIOR_GAME_WEIGHT * team_games_df['def_success_rate_allowed_prior']
      + team_games_df['prior_games'] * team_games_df['pregame_def_success_rate_allowed_current'].fillna(0)
    )
    / (PRIOR_GAME_WEIGHT + team_games_df['prior_games'])
)


In [38]:
team_games_df.columns

Index(['season', 'week', 'gameday', 'game_id', 'team', 'opponent', 'off_epa',
       'off_success_rate', 'pass_epa', 'rush_epa', 'def_epa_allowed',
       'def_success_rate_allowed', 'points_scored', 'points_allowed', 'win',
       'turnovers', 'takeaways', 'off_sack_rate', 'def_sack_rate', 'rest_days',
       'is_home', 'neutral_site', 'off_epa_last3', 'off_success_rate_last3',
       'pass_epa_last3', 'rush_epa_last3', 'def_epa_allowed_last3',
       'def_success_rate_allowed_last3', 'off_epa_last5',
       'off_success_rate_last5', 'pass_epa_last5', 'rush_epa_last5',
       'def_epa_allowed_last5', 'def_success_rate_allowed_last5',
       'off_epa_ewm', 'off_success_rate_ewm', 'pass_epa_ewm', 'rush_epa_ewm',
       'def_epa_allowed_ewm', 'def_success_rate_allowed_ewm',
       'off_epa_previous_season', 'off_success_rate_previous_season',
       'pass_epa_previous_season', 'rush_epa_previous_season',
       'def_epa_allowed_previous_season',
       'def_success_rate_allowed_previous_

In [39]:
INITIAL_ELO = 1500
K_FACTOR = 20
HOME_ELO_BONUS = 50

In [40]:
ratings = {}
elo_rows = []
elo_team_rows = []

In [41]:
elo_games = (
    completed_schedule
    .sort_values(['gameday', 'game_id'])
    .reset_index(drop=True)
)
for game in elo_games.itertuples(index=False):
    home_team = game.home_team
    away_team = game.away_team
    home_elo = ratings.get(home_team, INITIAL_ELO)
    away_elo = ratings.get(away_team, INITIAL_ELO)  
    effective_home_elo = home_elo
    
    if not game.neutral_site:
        effective_home_elo += HOME_ELO_BONUS
    expected_home = 1 / (1 + 10 ** ((away_elo - effective_home_elo) / 400))

    elo_rows.append({
    'game_id': game.game_id,
    'home_elo': home_elo,
    'away_elo': away_elo,
    'elo_diff': home_elo - away_elo,
    'elo_expected_home_win': expected_home,
})
    elo_team_rows.append({
    'game_id': game.game_id,
    'team': home_team,
    'pregame_elo': home_elo,
    'opponent_pregame_elo': away_elo,
})
    elo_team_rows.append({
    'game_id': game.game_id,
    'team': away_team,
    'pregame_elo': away_elo,
    'opponent_pregame_elo': home_elo,
})
    
    if game.home_score > game.away_score:
        home_result = 1.0
    elif game.home_score < game.away_score:
        home_result = 0.0
    else:
        home_result = 0.5

    elo_change = K_FACTOR * (home_result - expected_home)
    
    ratings[home_team] = home_elo + elo_change
    ratings[away_team] = away_elo - elo_change




In [42]:
elo_team_df = pd.DataFrame(elo_team_rows)
elo_df = pd.DataFrame(elo_rows)

In [43]:
team_games_df = team_games_df.merge(
    elo_team_df,
    on=['game_id', 'team'],
    how='left',
)

In [44]:
team_games_df['pregame_sos'] = (
    team_games_df
    .groupby(['season', 'team'])['opponent_pregame_elo']
    .transform(lambda x: x.shift(1).expanding().mean())
    )

In [45]:
team_games_df.columns

Index(['season', 'week', 'gameday', 'game_id', 'team', 'opponent', 'off_epa',
       'off_success_rate', 'pass_epa', 'rush_epa', 'def_epa_allowed',
       'def_success_rate_allowed', 'points_scored', 'points_allowed', 'win',
       'turnovers', 'takeaways', 'off_sack_rate', 'def_sack_rate', 'rest_days',
       'is_home', 'neutral_site', 'off_epa_last3', 'off_success_rate_last3',
       'pass_epa_last3', 'rush_epa_last3', 'def_epa_allowed_last3',
       'def_success_rate_allowed_last3', 'off_epa_last5',
       'off_success_rate_last5', 'pass_epa_last5', 'rush_epa_last5',
       'def_epa_allowed_last5', 'def_success_rate_allowed_last5',
       'off_epa_ewm', 'off_success_rate_ewm', 'pass_epa_ewm', 'rush_epa_ewm',
       'def_epa_allowed_ewm', 'def_success_rate_allowed_ewm',
       'off_epa_previous_season', 'off_success_rate_previous_season',
       'pass_epa_previous_season', 'rush_epa_previous_season',
       'def_epa_allowed_previous_season',
       'def_success_rate_allowed_previous_

In [46]:
team_games_df['short_rest'] = (
    team_games_df['rest_days'] < 6
).astype(int)

In [47]:
opponent_rest = (
    team_games_df[['game_id', 'team', 'rest_days']]
    .rename(columns=
            {'team': 'opponent',
            'rest_days': 'opponent_rest_days'})
)
team_games_df = team_games_df.merge(
    opponent_rest,
    on=['game_id', 'opponent'],
    how='left',
)

In [48]:
team_games_df['rest_diff'] = (
    team_games_df['rest_days'] - team_games_df['opponent_rest_days']
)

In [49]:
team_games_df['pregame_turnovers'] = (
    team_games_df
    .groupby(['season', 'team'])['turnovers']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['pregame_takeaways'] = (
    team_games_df
    .groupby(['season', 'team'])['takeaways']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['pregame_off_sack_rate'] = (
    team_games_df
    .groupby(['season', 'team'])['off_sack_rate']
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_games_df['pregame_def_sack_rate'] = (
    team_games_df
    .groupby(['season', 'team'])['def_sack_rate']
    .transform(lambda x: x.shift(1).expanding().mean())
)

## QB section

In [50]:
qb_games = (
    pbp.loc[pbp['qb_dropback'].eq(1) & pbp['passer_player_id'].notna()]
    .groupby(['game_id', 'passer_player_id'], as_index=False)
    .agg(
        qb_epa = ('epa', 'mean'),
        qb_success_rate = ('success', 'mean'))
    .merge(
        completed_schedule[['game_id', 'gameday']],
        on='game_id',
        validate='many_to_one')
    .sort_values(['passer_player_id', 'gameday'])
)

In [51]:
qb_group = qb_games.groupby('passer_player_id')

qb_games['pregame_qb_epa'] = qb_group['qb_epa'].transform(
    lambda x: x.shift(1).expanding().mean())

qb_games['pregame_qb_epa_last5'] = qb_group['qb_epa'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean())

qb_games['pregame_qb_success_rate'] = qb_group['qb_success_rate'].transform(
    lambda x: x.shift(1).expanding().mean())

In [52]:
starters = pd.concat([
    completed_schedule[['game_id', 'home_team', 'home_qb_id', 'home_qb_name']]
    .rename(columns={
        'home_team': 'team', 
        'home_qb_id': 'starting_qb_id', 
        'home_qb_name': 'starting_qb_name'}),
    completed_schedule[['game_id', 'away_team', 'away_qb_id', 'away_qb_name']]
    .rename(columns={
        'away_team': 'team', 
        'away_qb_id': 'starting_qb_id', 
        'away_qb_name': 'starting_qb_name'})
], ignore_index=True)

starters = starters.merge(
    qb_games[[
        'game_id',
        'passer_player_id',
        'pregame_qb_epa',
        'pregame_qb_epa_last5',
        'pregame_qb_success_rate'
    ]],
    left_on=['game_id', 'starting_qb_id'],
    right_on=['game_id', 'passer_player_id'],
    how='left',
    validate='one_to_one'
).drop(columns=['passer_player_id'])

In [53]:
team_games_df = team_games_df.merge(
    starters,
    on=['game_id', 'team'],
    how='left',
    validate='one_to_one'
)

In [54]:
team_games_df.columns

Index(['season', 'week', 'gameday', 'game_id', 'team', 'opponent', 'off_epa',
       'off_success_rate', 'pass_epa', 'rush_epa', 'def_epa_allowed',
       'def_success_rate_allowed', 'points_scored', 'points_allowed', 'win',
       'turnovers', 'takeaways', 'off_sack_rate', 'def_sack_rate', 'rest_days',
       'is_home', 'neutral_site', 'off_epa_last3', 'off_success_rate_last3',
       'pass_epa_last3', 'rush_epa_last3', 'def_epa_allowed_last3',
       'def_success_rate_allowed_last3', 'off_epa_last5',
       'off_success_rate_last5', 'pass_epa_last5', 'rush_epa_last5',
       'def_epa_allowed_last5', 'def_success_rate_allowed_last5',
       'off_epa_ewm', 'off_success_rate_ewm', 'pass_epa_ewm', 'rush_epa_ewm',
       'def_epa_allowed_ewm', 'def_success_rate_allowed_ewm',
       'off_epa_previous_season', 'off_success_rate_previous_season',
       'pass_epa_previous_season', 'rush_epa_previous_season',
       'def_epa_allowed_previous_season',
       'def_success_rate_allowed_previous_

## Final Features (pre-testing)

In [55]:
feature_columns = [
    'pregame_off_epa_blended',
    'pregame_off_success_rate_blended',
    'pregame_pass_epa_blended',
    'pregame_rush_epa_blended',
    'pregame_def_epa_allowed_blended',
    'pregame_def_success_rate_allowed_blended',
    'off_epa_last3',
    'off_success_rate_last3',
    'pass_epa_last3',
    'rush_epa_last3',
    'def_epa_allowed_last3',
    'def_success_rate_allowed_last3',
    'off_epa_last5',
    'off_success_rate_last5',
    'pass_epa_last5',
    'rush_epa_last5',
    'def_epa_allowed_last5',
    'def_success_rate_allowed_last5',
    'off_epa_ewm',
    'off_success_rate_ewm',
    'pass_epa_ewm',
    'rush_epa_ewm',
    'def_epa_allowed_ewm',
    'def_success_rate_allowed_ewm',
    'pregame_elo',
    'pregame_sos',
    'rest_days',
    'short_rest',
    'prior_games',
    'pregame_turnovers',
    'pregame_takeaways',
    'pregame_off_sack_rate',
    'pregame_def_sack_rate',
    'pregame_qb_epa',
    'pregame_qb_epa_last5',
    'pregame_qb_success_rate'
]

In [56]:
home_features = team_games_df.loc[
    team_games_df['is_home'].eq(1),
    ['game_id', 'team', *feature_columns],
].copy()

home_features = home_features.rename(
    columns={'team': 'home_team',
             **{column: f'home_{column}' for column in feature_columns}}
)

away_features = team_games_df.loc[
    team_games_df['is_home'].eq(0),
    ['game_id', 'team', *feature_columns],
].copy()

away_features = away_features.rename(
    columns={'team': 'away_team',
             **{column: f'away_{column}' for column in feature_columns}}
)

In [57]:
games = completed_schedule.loc[
    completed_schedule['home_score'].ne(completed_schedule['away_score']),
    [
        'season',
        'week',
        'gameday',
        'game_id',
        'home_team',
        'away_team',
        'home_score',
        'away_score',
        'neutral_site'
    ],
].copy()
games['home_win'] = (games['home_score'] > games['away_score']).astype(int)

In [58]:
model_df = games.merge(
    home_features,
    on=['game_id', 'home_team'],
    how='left',
    validate='one_to_one'
).merge(
    away_features,
    on=['game_id', 'away_team'],
    how='left',
    validate='one_to_one'
)


In [59]:
model_df = model_df.drop(
    columns=['home_score', 'away_score']
)

In [60]:
model_df.columns

Index(['season', 'week', 'gameday', 'game_id', 'home_team', 'away_team',
       'neutral_site', 'home_win', 'home_pregame_off_epa_blended',
       'home_pregame_off_success_rate_blended',
       'home_pregame_pass_epa_blended', 'home_pregame_rush_epa_blended',
       'home_pregame_def_epa_allowed_blended',
       'home_pregame_def_success_rate_allowed_blended', 'home_off_epa_last3',
       'home_off_success_rate_last3', 'home_pass_epa_last3',
       'home_rush_epa_last3', 'home_def_epa_allowed_last3',
       'home_def_success_rate_allowed_last3', 'home_off_epa_last5',
       'home_off_success_rate_last5', 'home_pass_epa_last5',
       'home_rush_epa_last5', 'home_def_epa_allowed_last5',
       'home_def_success_rate_allowed_last5', 'home_off_epa_ewm',
       'home_off_success_rate_ewm', 'home_pass_epa_ewm', 'home_rush_epa_ewm',
       'home_def_epa_allowed_ewm', 'home_def_success_rate_allowed_ewm',
       'home_pregame_elo', 'home_pregame_sos', 'home_rest_days',
       'home_short_rest

### Val Set

In [61]:
features = [
    f"{side}_{column}"
    for side in ["home", "away"]
    for column in feature_columns
] + ["neutral_site"]

In [62]:
train = model_df[model_df['season'].between(MODEL_START_SEASON, 2023)].copy()
val = model_df[model_df['season'] == 2024].copy()
test = model_df[model_df['season'] == 2025].copy()

y_train = train['home_win']
y_val = val['home_win']
y_test = test['home_win']

x_train = train[features]
x_val = val[features]
x_test = test[features]

In [63]:
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000))
])

In [64]:
model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](73,)","['home_pregame_off_epa_blended','home_pregame_off_success_rate_blended', 'home_pregame_pass_epa_blended',...,'away_pregame_qb_epa_last5', 'away_pregame_qb_success_rate','neutral_site']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,73
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhe

In [65]:
val_prob = model.predict_proba(x_val)[:, 1]
val_pred = (val_prob >= 0.5).astype(int)

v2_accuracy = accuracy_score(y_val, val_pred)
v2_logloss = log_loss(y_val, val_prob)
v2_brier = brier_score_loss(y_val, val_prob)
v2_auc = roc_auc_score(y_val, val_prob)

In [66]:
v2_val_results = pd.DataFrame({
    'model': ['logistic_regression'],
    'accuracy': [ v2_accuracy],
    'log_loss': [v2_logloss],
    'brier_score': [v2_brier],
    'roc_auc': [v2_auc]
})
v2_val_results.head()

,model,accuracy,log_loss,brier_score,roc_auc
0,logistic_regression,0.652632,0.622124,0.216818,0.709253


### Test Set

In [67]:
v2_test_prob = model.predict_proba(x_test)[:, 1]

v2_test_pred = (v2_test_prob >= 0.5).astype(int)

In [68]:
v2_test_accuracy = accuracy_score(y_test, v2_test_pred)
v2_test_logloss = log_loss(y_test, v2_test_prob)
v2_test_brier = brier_score_loss(y_test, v2_test_prob)
v2_test_auc = roc_auc_score(y_test, v2_test_prob)

In [69]:
test_results = pd.DataFrame({
    'model': ['test_model1'],
    'accuracy': [v2_test_accuracy],
    'log_loss': [v2_test_logloss],
    'brier_score': [v2_test_brier],
    'roc_auc': [v2_test_auc]
})
test_results.head()

,model,accuracy,log_loss,brier_score,roc_auc
0,test_model1,0.644366,0.647834,0.227508,0.676087


In [70]:
model_path = Path('../models/logistic_regression_v2.joblib')

artifact = {
    'model': model,
    'features': features,
    'target': 'home_win',
    'decision_threshold': 0.5,
    'training_seasons': '2010-2023',
    'validation_season': 2024,
    'test_season': 2025,
}

joblib.dump(artifact, model_path)

print(f'Saved model to: {model_path.resolve()}')

Saved model to: C:\Coding Projects\NFL-2026-Machine-Learning-Project\models\logistic_regression_v2.joblib


In [71]:
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

assert (project_root / "notebooks").exists(), (
    f"Could not identify project root from: {project_root}"
)

output_directory = project_root / "data" / "processed"
output_directory.mkdir(parents=True, exist_ok=True)

output_path = output_directory / "model_dataset_v2.csv"

model_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8",
)

print(f"Saved {len(model_df):,} rows to:")
print(output_path)

Saved 4,617 rows to:
C:\Coding Projects\NFL-2026-Machine-Learning-Project\data\processed\model_dataset_v2.csv
